In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import random

spark = SparkSession.builder.appName("TransitLoad").getOrCreate()
print("Spark version:", spark.version)

Spark version: 4.0.4


In [3]:
random.seed(42)  # makes the random numbers repeatable

stops = [f"S{i:02d}" for i in range(1, 11)]
routes = ["R1", "R2"]
capacity = 60  # seats + standing room per bus

rows = []
for route in routes:
    for t in range(1, 21):
        trip_id = f"{route}_T{t:02d}"
        hour = random.choice([7, 8, 9, 12, 17, 18, 19])
        for seq, stop in enumerate(stops, start=1):
            boardings = random.randint(0, 25)
            alightings = random.randint(0, 20)
            rows.append((route, trip_id, hour, seq, stop, boardings, alightings, capacity))

columns = ["route_id", "trip_id", "hour", "stop_seq", "stop_id",
           "boardings", "alightings", "capacity"]

df = spark.createDataFrame(rows, columns)
print("Total rows:", df.count())
df.show(5)

Total rows: 400
+--------+-------+----+--------+-------+---------+----------+--------+
|route_id|trip_id|hour|stop_seq|stop_id|boardings|alightings|capacity|
+--------+-------+----+--------+-------+---------+----------+--------+
|      R1| R1_T01|  18|       1|    S01|        3|         0|      60|
|      R1| R1_T01|  18|       2|    S02|       23|         8|      60|
|      R1| R1_T01|  18|       3|    S03|        7|         7|      60|
|      R1| R1_T01|  18|       4|    S04|        4|         3|      60|
|      R1| R1_T01|  18|       5|    S05|       21|        17|      60|
+--------+-------+----+--------+-------+---------+----------+--------+
only showing top 5 rows


In [5]:
df.printSchema()

root
 |-- route_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- hour: long (nullable = true)
 |-- stop_seq: long (nullable = true)
 |-- stop_id: string (nullable = true)
 |-- boardings: long (nullable = true)
 |-- alightings: long (nullable = true)
 |-- capacity: long (nullable = true)



In [7]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = (Window.partitionBy("trip_id")
           .orderBy("stop_seq")
           .rowsBetween(Window.unboundedPreceding, Window.currentRow))

df_load = (df
    .withColumn("onboard_raw", F.sum(F.col("boardings") - F.col("alightings")).over(w))
    .withColumn("onboard", F.greatest(F.col("onboard_raw"), F.lit(0)))
    .withColumn("load_factor", F.round(F.col("onboard") / F.col("capacity"), 2))
    .withColumn("overloaded", F.col("load_factor") > 0.85)
)

df_load.select("trip_id", "stop_seq", "stop_id", "onboard", "load_factor", "overloaded").show(15)

+-------+--------+-------+-------+-----------+----------+
|trip_id|stop_seq|stop_id|onboard|load_factor|overloaded|
+-------+--------+-------+-------+-----------+----------+
| R1_T01|       1|    S01|      3|       0.05|     false|
| R1_T01|       2|    S02|     18|        0.3|     false|
| R1_T01|       3|    S03|     18|        0.3|     false|
| R1_T01|       4|    S04|     19|       0.32|     false|
| R1_T01|       5|    S05|     23|       0.38|     false|
| R1_T01|       6|    S06|      7|       0.12|     false|
| R1_T01|       7|    S07|     19|       0.32|     false|
| R1_T01|       8|    S08|     17|       0.28|     false|
| R1_T01|       9|    S09|     16|       0.27|     false|
| R1_T01|      10|    S10|     13|       0.22|     false|
| R1_T02|       1|    S01|     11|       0.18|     false|
| R1_T02|       2|    S02|     13|       0.22|     false|
| R1_T02|       3|    S03|     18|        0.3|     false|
| R1_T02|       4|    S04|     24|        0.4|     false|
| R1_T02|     

In [9]:
overloaded = df_load.filter(F.col("overloaded"))
print("Overloaded stop records:", overloaded.count())
overloaded.select("route_id", "trip_id", "stop_id", "onboard", "load_factor").show(10)

Overloaded stop records: 32
+--------+-------+-------+-------+-----------+
|route_id|trip_id|stop_id|onboard|load_factor|
+--------+-------+-------+-------+-----------+
|      R1| R1_T02|    S08|     56|       0.93|
|      R1| R1_T02|    S09|     58|       0.97|
|      R1| R1_T02|    S10|     56|       0.93|
|      R1| R1_T09|    S05|     54|        0.9|
|      R1| R1_T09|    S06|     67|       1.12|
|      R1| R1_T09|    S07|     60|        1.0|
|      R1| R1_T09|    S08|     79|       1.32|
|      R1| R1_T09|    S09|     80|       1.33|
|      R1| R1_T09|    S10|     61|       1.02|
|      R2| R2_T03|    S06|     55|       0.92|
+--------+-------+-------+-------+-----------+
only showing top 10 rows


In [11]:
summary = (df_load
    .groupBy("route_id", "hour")
    .agg(
        F.round(F.avg("load_factor"), 2).alias("avg_load_factor"),
        F.round(F.max("load_factor"), 2).alias("max_load_factor"),
        F.sum(F.col("overloaded").cast("int")).alias("overloaded_count")
    )
    .orderBy(F.desc("avg_load_factor"))
)

summary.show()

+--------+----+---------------+---------------+----------------+
|route_id|hour|avg_load_factor|max_load_factor|overloaded_count|
+--------+----+---------------+---------------+----------------+
|      R2|   8|           0.53|            1.0|               1|
|      R1|   7|           0.48|           1.33|               9|
|      R2|  17|           0.43|            1.3|               7|
|      R2|  19|           0.43|           1.33|               5|
|      R2|   9|            0.4|           1.38|               4|
|      R2|  12|           0.36|           1.23|               6|
|      R1|   8|           0.34|           0.62|               0|
|      R2|  18|           0.28|           0.73|               0|
|      R1|  18|           0.21|           0.57|               0|
|      R1|   9|           0.19|           0.52|               0|
|      R1|  12|            0.1|           0.47|               0|
|      R1|  17|           0.08|           0.57|               0|
|      R2|   7|          

In [13]:
busiest = summary.first()
print(f"Busiest: Route {busiest['route_id']} at {busiest['hour']}:00 "
      f"(avg load factor {busiest['avg_load_factor']})")

Busiest: Route R2 at 8:00 (avg load factor 0.53)


In [14]:
summary.toPandas().to_csv("route_hour_summary.csv", index=False)
print("Saved route_hour_summary.csv")

Saved route_hour_summary.csv
